In [9]:
##################################################################################
################# TASK 1 : Basic Port Scanner with Socket ########################
################# Implements a simple TCP port scanner using the socket library ##
################# To check whether ports are open on a target host.  #############
##################################################################################

import socket

# Function to scan a single port
def scan_port(host, port):

    # To create a TCP socket
    tcp_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    #  To set timeout to 1 second
    tcp_socket.settimeout(1)

    # To try to connect to the port using connect_ex()
    # and return 0 if connection is successful
    try:
        result = tcp_socket.connect_ex((host, port))

        # To open port if result is 0
        if result == 0:
            return True
        else:
            return False
    
    # For Errors
    # For connection taking too long
    except socket.timeout:
        print(f"Timeout: Port {port} taking too long to respond")

    # For General socket errors (connection refused, network unreachable)
    except socket.error as e:
        print(f"Socket error on port {port}: {e}")
        return False
    
    # To catch any othe runexpected errors
    except Exception as e:
        print(f"Unexpected error on port {port}: {e}")
        return False
    
    # To close the socket always
    finally:
        tcp_socket.close()

# ------------------------
# Testing the Function
# ------------------------
# Target host
host = "scanme.nmap.org"

# To scan ports 20 to 1024
print (f"Scanning ports 20-1024 on {host}.......\n")

for port in range(20, 1025):
    if scan_port(host, port):
        print(f"Port {port} is OPEN")

Scanning ports 20-1024 on scanme.nmap.org.......

Port 22 is OPEN
Port 80 is OPEN


In [6]:
############################################################################################################
#################       TASK 2 : Basic Port Scanner with Socket and Banner Grabbing ########################
###### Implements a simple TCP port scanner using the socket library #######################################
###### To check whether ports are open on a target host.  ##################################################
###### And a banner grabbing, which attempts to read service information from open ports.###################
############################################################################################################

##################################################################################
################# TASK 1 : Basic Port Scanner with Socket ########################
################# Implements a simple TCP port scanner using the socket library ##
################# To check whether ports are open on a target host.  #############
##################################################################################

import socket

# Function to scan a single port
def scan_port(host, port):

    # To create a TCP socket
    tcp_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    #  To set timeout to 1 second
    tcp_socket.settimeout(1)

    # Default banner if none is received
    banner = ""
    is_open = False

    # To try to connect to the port using connect_ex()
    # and return 0 if connection is successful
    try:
        result = tcp_socket.connect_ex((host, port))

        # To open port if result is 0
        if result == 0:
            is_open = True
        
            try:
                # To set timeout for banner grabbing to 0.5seconds
                tcp_socket.settimeout(0.5)

                # To try to receive up to 1024 bytes
                data = tcp_socket.recv(1024)

                #To convert bytes to string and ignore decoding errors
                banner = data.decode('utf-8', errors="ignore").strip()

            # timeout error for banner grabbibg
            except socket.timeout:
                banner = ""

            # For other socket issues while receiving banner
            except socket.error:
                banner =""
            
        else:
            is_open = False     # Port is closed, leave default
            banner = ""
            
        return is_open, banner

    
    # For Errors Handling
    # For connection taking too long
    except socket.timeout:
        print(f"Timeout: Port {port} taking too long to respond")
        return False, ""

    # For General socket errors (connection refused, network unreachable)
    except socket.error as e:
        print(f"Socket error on port {port}: {e}")
        return False, ""
    
    # To catch any other runexpected errors
    except Exception as e:
        print(f"Unexpected error on port {port}: {e}")
        return False, ""
    
    # To close the socket always
    finally:
        tcp_socket.close()

    return False, ""

# ------------------------
# Testing the Function
# ------------------------
# Target host
host = "scanme.nmap.org"

print (f"Scanning ports 20-1024 on {host}.......\n")

# To scan ports 20 to 1024
for port in range(20, 1025):

    is_open, banner = scan_port(host, port)

    # Print outputs
    if is_open:
        if banner:
            print(f"Port {port} is OPEN | Banner: {banner}")
        else:
            print(f"Port {port} is OPEN | No banner received")
        


Scanning ports 20-1024 on scanme.nmap.org.......

Port 22 is OPEN | Banner: SSH-2.0-OpenSSH_6.6.1p1 Ubuntu-2ubuntu2.13
Port 80 is OPEN | No banner received


In [3]:
############################################################################################################
###############################  TASK 3 : NMAP Port Scanner ################################################
#########  To use the Nmap Python library:                  ################################################
# To perform a more advanced scan that identifies open ports along with their services and versions, ports.#
############################################################################################################

import nmap  # pip install python-nmap

def nmap_scan(host, port_range='1-1024'):

    results = []  # List to store port info

    try:
        # Create a PortScanner object
        my_nmap = nmap.PortScanner()

        # Perform a service/version scan (-sV) on the target
        my_nmap.scan(hosts=host, ports=port_range, arguments='-sV')

        # Iterate over all hosts (usually one)
        for h in my_nmap.all_hosts():

            if my_nmap[h].state() == "up":
                print(f"{h} is up")

                # Iterate over all protocols (tcp, udp)
                for proto in my_nmap[h].all_protocols():
                    ports = my_nmap[h][proto].keys()  # Get all scanned ports for this protocol
                    for port in ports:
                        port_info = my_nmap[h][proto][port]
                        # Collect info about open ports
                        if port_info['state'] == 'open':
                            results.append({
                                'port': port,
                                'state': port_info['state'],
                                'service': port_info.get('name', ''),
                                'version': port_info.get('version', '')
                            })

    except nmap.PortScannerError as e:
        print(f"Nmap error: {e}")

    except Exception as e:
        print(f"Unexpected error: {e}")

    return results


# ------------------------
# Testing the Function
# ------------------------
host = "scanme.nmap.org"

open_ports = nmap_scan(host, port_range='20-1024')

print(f"Open ports on {host}:")
for port in open_ports:
    print(f"Port {port['port']} | State: {port['state']} | Service: {port['service']} | Version: {port['version']}")

45.33.32.156 is up
Open ports on scanme.nmap.org:
Port 22 | State: open | Service: ssh | Version: 6.6.1p1 Ubuntu 2ubuntu2.13
Port 80 | State: open | Service: http | Version: 2.4.7


In [4]:
####################################################################################################################
#######        TASK 4: Shodan OSINT Lookup #########################################################################
#T0 performs an OSINT lookup using the Shodan API to gather publicly available information about an IP address, ####
# such as organization, operating system, country, open ports, and service banners.         ########################
####################################################################################################################

import shodan

def shodan_lookup(ip, api_key):
    """
    Look up information about an IP address using Shodan.

    Returns a dictionary with:
    ip, org, os, country, ports, banners
    """

    # Dictionary to store the results
    result_dict = {
        'ip': ip,
        'org': '',
        'os': '',
        'country': '',
        'ports': [],
        'banners': []
    }

    try:
        # 1. Initialize the Shodan API using the API key
        api = shodan.Shodan(api_key)

        # 2. Get information about the IP address
        result = api.host(ip)

        # 3. Extract main fields from the result
        result_dict['org'] = result.get('org', '')
        result_dict['os'] = result.get('os', '')
        result_dict['country'] = result.get('country_name', '')
        result_dict['ports'] = result.get('ports', [])

        # 4. Extract banners from result['data']
        # Each item in 'data' contains service information
        for item in result.get('data', []):
            banner = item.get('data', '').strip()
            if banner:
                result_dict['banners'].append(banner)

    # 5. Handle Shodan API errors
    except shodan.APIError as e:
        print(f"Shodan API error: {e}")

    return result_dict


# ------------------------
# Testing the Function
# ------------------------

# Replace with your Shodan API key
API_KEY = "mW3Zajxc1LHrdG7UBYcUK7F5PpMbSPTG"

# Public IP to test
ip = "8.8.8.8"

info = shodan_lookup(ip, API_KEY)

print("Shodan Lookup Result:\n")

print("IP:", info['ip'])
print("Organization:", info['org'])
print("Operating System:", info['os'])
print("Country:", info['country'])
print("Open Ports:", info['ports'])

print("\nService Banners:")
for banner in info['banners']:
    print(banner[:100])  # print first 100 characters only

Shodan Lookup Result:

IP: 8.8.8.8
Organization: Google LLC
Operating System: None
Country: United States
Open Ports: [443, 53]

Service Banners:
Recursion: enabled
Recursion: enabled
HTTP/1.1 200 OK
Content-Security-Policy: object-src 'none';base-uri 'self';script-src 'nonce-xOZgEz
